# UATimer — Kaggle Notebook (self-contained)

Runnable end-to-end on **Kaggle** with no repository files, no C compiler,
and no internet. It bundles a pure-Python port of the UATimer policy, the
lightweight LR baseline, a trace generator/replay, and the full
measurement-analysis pipeline (stats, figures, LaTeX tables).

Companion to the paper *"UATimer: Lightweight Context-Aware Power
Management for Heterogeneous IoT Systems"* and the C reference
implementation at `github.com/leemgs/uatimer`.

---
### How to use on Kaggle
1. **Create → Notebook**, then either upload this `.ipynb`
   (*File → Import Notebook*) or paste the cells.
2. *(optional)* To analyse **your own measurements**, add a Kaggle
   Dataset containing `raw_runs.csv` (schema below) via
   *Add Data*; the notebook auto-detects it under `/kaggle/input/`.
3. *Run All*. Figures and LaTeX tables are written to `/kaggle/working/`
   and appear in the notebook's **Output** tab.

> ⚠️ **No fabricated data.** Without an attached dataset the notebook runs
> in **synthetic-demo** mode: the numbers are random illustration to make
> the pipeline runnable — **not** the paper's measurements. With a real
> `raw_runs.csv` attached, it reports only what your data contains.

### Raw data schema (`raw_runs.csv`, one row per run)
`platform, workload, policy, run, energy_mj, delay_ms, frame_drop_pct, jitter_ms`
- `policy` ∈ {Baseline, Fixed, LR, UATimer}; ≥10 runs per condition recommended
- leave `frame_drop_pct`/`jitter_ms` empty except for AR/VR

## 0. Environment & output paths (Kaggle-aware)

In [ ]:
import os, glob
import numpy as np, pandas as pd

# Kaggle writes to /kaggle/working and mounts datasets under /kaggle/input.
ON_KAGGLE = os.path.isdir("/kaggle")
OUT_DIR   = "/kaggle/working" if ON_KAGGLE else "./uatimer_out"
os.makedirs(OUT_DIR, exist_ok=True)

# Look for an attached raw_runs.csv anywhere under /kaggle/input.
def find_raw_csv():
    for base in ("/kaggle/input", "."):
        hits = glob.glob(os.path.join(base, "**", "raw_runs.csv"), recursive=True)
        if hits:
            return sorted(hits)[0]
    return None

RAW_CSV = find_raw_csv()
USE_SYNTHETIC_DEMO = RAW_CSV is None   # auto: real data if present, else demo

print("on kaggle:", ON_KAGGLE)
print("output dir:", OUT_DIR)
print("raw CSV:", RAW_CSV)
print("mode:", "SYNTHETIC DEMO" if USE_SYNTHETIC_DEMO else "REAL DATA")

CONF_LEVEL   = 0.95
POLICY_ORDER = ["Baseline", "Fixed", "LR", "UATimer"]
WORKLOADS    = ["Web Browsing","Video Playback","AI Inference",
                "AR/VR Gaming","IoT Sensing"]

## 1. Pure-Python UATimer policy core

A faithful port of the C reference implementation (`src/uatimer.c`):
the EWMA idle estimate (Eq. 3), per-mode scale `rho` (Eq. 7), the
effective threshold with break-even floor (Eqs. 2, 6), and the
latency-critical inhibit. No compilation required.

In [ ]:
from dataclasses import dataclass, field

MODE_ENERGY, MODE_BALANCED, MODE_LATENCY = 0, 1, 2

@dataclass
class ModeCfg:
    w_e: float; w_d: float; t_min: int; inhibit: bool

@dataclass
class UAConfig:
    lambda_: float = 0.5
    alpha:   float = 0.5
    t_hat0:  int   = 1000
    t_be:    int   = 50
    t_max:   int   = 30000
    modes: list = field(default_factory=lambda: [
        ModeCfg(0.8, 0.2, 50,  False),   # energy-conservative
        ModeCfg(0.5, 0.5, 50,  False),   # balanced
        ModeCfg(0.2, 0.8, 100, True),    # latency-critical
    ])

class UATimer:
    """Port of src/uatimer.c. Times are in milliseconds."""
    def __init__(self, cfg: UAConfig, now_ms: int = 0):
        self.cfg = cfg
        self.t_hat = float(cfg.t_hat0)
        self.t_last = now_ms
        self.mode = MODE_BALANCED
        self.armed = False
        self.n_events = self.n_suspends = self.n_inhibited = 0
        self.total_sleep_ms = 0

    def mode_scale(self, mode=None):
        m = self.cfg.modes[self.mode if mode is None else mode]
        rho = 1.0 + self.cfg.alpha * (m.w_d - m.w_e)
        return max(rho, 0.0)

    def effective_threshold(self):
        m = self.cfg.modes[self.mode]
        scaled = int(self.mode_scale() * self.t_hat + 0.5)
        floor = max(m.t_min, self.cfg.t_be)          # never below break-even
        return min(max(scaled, floor), self.cfg.t_max)

    def on_event(self, now_ms):
        idle = max(now_ms - self.t_last, 0)
        self.t_last = now_ms
        self.n_events += 1
        lam = self.cfg.lambda_
        self.t_hat = lam * self.t_hat + (1.0 - lam) * idle   # Eq. (3)
        self.armed = not self.cfg.modes[self.mode].inhibit

    def set_mode(self, mode):
        self.mode = mode
        self.armed = not self.cfg.modes[self.mode].inhibit

    def on_expiry(self, sleep_ms):
        if not self.armed or self.cfg.modes[self.mode].inhibit:
            self.n_inhibited += 1
            return False
        self.armed = False
        self.n_suspends += 1
        if sleep_ms > 0:
            self.total_sleep_ms += sleep_ms
        return True

# quick self-check of the properties the paper states
_cfg = UAConfig()
_t = UATimer(_cfg)
assert _t.mode_scale(MODE_ENERGY) < _t.mode_scale(MODE_BALANCED) < _t.mode_scale(MODE_LATENCY)
assert abs(_t.mode_scale(MODE_BALANCED) - 1.0) < 1e-9
print("policy core OK; mode scales:",
      [round(_t.mode_scale(m),3) for m in (MODE_ENERGY,MODE_BALANCED,MODE_LATENCY)])

## 2. LR baseline + trace generator + replay

Pure-Python ports of `src/lr_baseline.c`, `tools/gen_trace.py`, and
`src/replay.c`. The replay is a **functional** model of the control logic
(transition counts, sleep time); it does **not** model device power and
does not produce the paper's energy (mJ) values.

In [ ]:
# --- LR predictor (offline OLS fit; 4 coefficients) ---
class LR:
    def __init__(self): self.coef=[1.0,0.0,0.0,0.0]; self.hist=[0.0,0.0,0.0]
    def predict(self):
        y=self.coef[3]+self.coef[0]*self.hist[0]+self.coef[1]*self.hist[1]+self.coef[2]*self.hist[2]
        return max(y,0.0)
    def observe(self,x): self.hist=[x,self.hist[0],self.hist[1]]
    def fit(self,idle):
        idle=np.asarray(idle,float)
        if len(idle)<5: return
        X=np.column_stack([idle[2:-1],idle[1:-2],idle[0:-3],np.ones(len(idle)-3)])
        y=idle[3:]
        self.coef=list(np.linalg.lstsq(X+1e-9,y,rcond=None)[0])

# --- synthetic trace shapes (illustrative, not measured) ---
def gen_traces(seed=42):
    rng=np.random.default_rng(seed); out={}
    web=[]
    while len(web)<2000:
        web+=[(int(rng.integers(80,400)),MODE_BALANCED) for _ in range(int(rng.integers(3,12)))]
        web.append((int(rng.integers(4000,30000)),MODE_BALANCED))
    out["web"]=web[:2000]
    out["video"]=[((int(rng.integers(30,60)) if rng.random()>0.05 else int(rng.integers(2000,8000))),MODE_BALANCED) for _ in range(2000)]
    out["sensing"]=[(10000+int(rng.integers(-200,200)),MODE_ENERGY) for _ in range(2000)]
    out["arvr"]=[(16+int(rng.integers(0,2)),MODE_LATENCY) for _ in range(2000)]
    return out

# --- replay UATimer / Fixed / LR over a trace ---
def replay_uatimer(cfg,trace):
    t=UATimer(cfg,0); clock=0; s=dict(events=0,suspends=0,profitable=0,unprofitable=0,sleep=0,idle=0)
    for d,mode in trace:
        if mode is not None: t.set_mode(mode)
        t_eff=t.effective_threshold(); s["idle"]+=d
        if not cfg.modes[t.mode].inhibit and d>t_eff:
            sleep=d-t_eff
            if t.on_expiry(sleep):
                s["suspends"]+=1
                s["profitable" if sleep>=cfg.t_be else "unprofitable"]+=1
                s["sleep"]+=sleep
        clock+=d; t.on_event(clock); s["events"]+=1
    return s

def replay_fixed(timeout,t_be,trace):
    s=dict(events=0,suspends=0,profitable=0,unprofitable=0,sleep=0,idle=0)
    for d,_ in trace:
        s["idle"]+=d
        if d>timeout:
            sl=d-timeout; s["suspends"]+=1
            s["profitable" if sl>=t_be else "unprofitable"]+=1; s["sleep"]+=sl
        s["events"]+=1
    return s

def replay_lr(t_be,trace):
    lr=LR(); lr.fit([d for d,_ in trace])
    s=dict(events=0,suspends=0,profitable=0,unprofitable=0,sleep=0,idle=0)
    for d,_ in trace:
        s["idle"]+=d; pred=lr.predict()
        if pred>=t_be:
            s["suspends"]+=1
            if d>=t_be: s["profitable"]+=1; s["sleep"]+=d
            else: s["unprofitable"]+=1
        lr.observe(d); s["events"]+=1
    return s

cfg=UAConfig()
traces=gen_traces()
rows=[]
for name,tr in traces.items():
    u=replay_uatimer(cfg,tr); f=replay_fixed(3000,cfg.t_be,tr); l=replay_lr(cfg.t_be,tr)
    for pol,st in [("UATimer",u),("Fixed",f),("LR",l)]:
        rows.append([name,pol,st["events"],st["suspends"],st["profitable"],
                     st["unprofitable"],st["sleep"],st["idle"],
                     round(100*st["sleep"]/st["idle"],1) if st["idle"] else 0])
replay_df=pd.DataFrame(rows,columns=["trace","policy","events","suspends",
              "profitable","unprofitable","sleep_ms","idle_ms","sleep_pct"])
print("Control-behaviour replay (NOT energy). AR/VR shows 0 UATimer suspends (inhibit):")
replay_df

## 3. Load or synthesize the raw per-run measurements

In [ ]:
REQUIRED=["platform","workload","policy","run","energy_mj","delay_ms","frame_drop_pct","jitter_ms"]

def make_demo(n_runs=10,seed=0):
    """Random illustrative data ONLY. Not measurements."""
    rng=np.random.default_rng(seed)
    base={"Web Browsing":(1200,30),"Video Playback":(1800,25),"AI Inference":(2200,28),
          "AR/VR Gaming":(2500,20),"IoT Sensing":(400,12)}
    emul={"Baseline":1.0,"Fixed":0.82,"LR":0.75,"UATimer":0.73}
    dmul={"Baseline":1.0,"Fixed":1.6,"LR":1.5,"UATimer":1.45}
    fdv={"Baseline":2.5,"Fixed":3.2,"LR":1.8,"UATimer":0.5}
    jtv={"Baseline":3.0,"Fixed":3.5,"LR":2.7,"UATimer":1.2}
    r=[]
    for wl,(e0,d0) in base.items():
        plat="iot_node" if wl=="IoT Sensing" else "phone"
        for pol in POLICY_ORDER:
            for i in range(1,n_runs+1):
                e=e0*emul[pol]*(1+rng.normal(0,0.04)); d=d0*dmul[pol]*(1+rng.normal(0,0.05))
                fd=jt=np.nan
                if wl=="AR/VR Gaming":
                    fd=fdv[pol]*(1+rng.normal(0,0.1)); jt=jtv[pol]*(1+rng.normal(0,0.1))
                r.append([plat,wl,pol,i,round(e,1),round(d,1),
                          round(fd,2) if fd==fd else np.nan, round(jt,2) if jt==jt else np.nan])
    return pd.DataFrame(r,columns=REQUIRED)

if USE_SYNTHETIC_DEMO:
    print("*** SYNTHETIC DEMO DATA — NOT MEASUREMENTS. Attach raw_runs.csv to use real data. ***")
    df=make_demo()
else:
    df=pd.read_csv(RAW_CSV); print("loaded real data from",RAW_CSV)

assert not (set(REQUIRED)-set(df.columns)), f"missing columns: {set(REQUIRED)-set(df.columns)}"
df["policy"]=pd.Categorical(df["policy"],POLICY_ORDER,ordered=True)
print(len(df),"runs")
df.head()

## 4. Data-quality checks

In [ ]:
counts=df.pivot_table(index="workload",columns="policy",values="run",aggfunc="count",observed=True)
display(counts)
n=counts.stack().dropna().unique()
print("runs per cell:",sorted(n))
if (counts.stack()<10).any(): print("WARNING: some cells <10 runs; weak dispersion/significance.")
assert df[["energy_mj","delay_ms"]].notna().all().all(),"missing energy/delay"
print("checks complete")

## 5. Aggregate: mean, std, 95% CI (Student-t)

In [ ]:
from scipy import stats
def agg(frame,col):
    g=frame.groupby(["workload","policy"],observed=True)[col].agg(["mean","std","count"]).reset_index()
    tcrit=g["count"].apply(lambda k: stats.t.ppf(0.5+CONF_LEVEL/2,k-1) if k>1 else np.nan)
    g["ci95"]=tcrit*g["std"]/np.sqrt(g["count"]); return g
energy_stats=agg(df,"energy_mj"); delay_stats=agg(df,"delay_ms")

def pivot_pm(s):
    s=s.copy(); s["cell"]=s.apply(lambda r:f"{r['mean']:.0f} ± {r['ci95']:.0f}",axis=1)
    p=s.pivot(index="workload",columns="policy",values="cell")
    return p.reindex([w for w in WORKLOADS if w in p.index])[[c for c in POLICY_ORDER if c in p.columns]]
print("Energy (mJ) mean ± 95% CI"); display(pivot_pm(energy_stats))
print("Delay (ms) mean ± 95% CI");  display(pivot_pm(delay_stats))

## 6. Energy reduction (propagated uncertainty)

In [ ]:
em=energy_stats.set_index(["workload","policy"])
def reduction(wl,comp):
    a=em.loc[(wl,"UATimer")]; b=em.loc[(wl,comp)]; ratio=a["mean"]/b["mean"]
    rel=np.sqrt((a["ci95"]/a["mean"])**2+(b["ci95"]/b["mean"])**2)
    return (1-ratio)*100, ratio*rel*100
rows=[]
for wl in [w for w in WORKLOADS if (w,"UATimer") in em.index]:
    row={"workload":wl}
    for c in ["Baseline","Fixed","LR"]:
        r,ci=reduction(wl,c); row[f"vs {c} (%)"]=f"{r:.1f} ± {ci:.1f}"
    rows.append(row)
red_df=pd.DataFrame(rows).set_index("workload"); display(red_df)
vs_base=[reduction(w,"Baseline")[0] for w in WORKLOADS if (w,"UATimer") in em.index]
print(f"Range vs Baseline: {min(vs_base):.1f}–{max(vs_base):.1f}%  (mean {np.mean(vs_base):.1f}%)")

## 7. Significance tests (Wilcoxon signed-rank)

In [ ]:
def paired(wl,pol):
    return df[(df.workload==wl)&(df.policy==pol)].sort_values("run")["energy_mj"].to_numpy()
rows=[]
for wl in [w for w in WORKLOADS if w in df.workload.unique()]:
    ua=paired(wl,"UATimer")
    for c in ["Baseline","Fixed","LR"]:
        cc=paired(wl,c); k=min(len(ua),len(cc))
        if k<3: rows.append([wl,c,"n<3","-",k]); continue
        try: st,p=stats.wilcoxon(ua[:k],cc[:k]); test="Wilcoxon"
        except ValueError: st,p=stats.mannwhitneyu(ua,cc,alternative="two-sided"); test="MannWhitneyU"
        rows.append([wl,c,test,f"{p:.4f}",k])
display(pd.DataFrame(rows,columns=["workload","vs","test","p_value","n"]))
print("With the demo's small n, p-values are illustrative only.")

## 8. Figures (saved to the output dir)

In [ ]:
import matplotlib
matplotlib.use("Agg") if not ON_KAGGLE else None
import matplotlib.pyplot as plt
es=energy_stats.set_index(["workload","policy"]); ds=delay_stats.set_index(["workload","policy"])

fig,ax=plt.subplots(figsize=(6,4.2)); mk={"Baseline":"s","Fixed":"^","LR":"D","UATimer":"o"}
for wl in [w for w in WORKLOADS if (w,"Baseline") in es.index]:
    e0=es.loc[(wl,"Baseline"),"mean"]
    ax.plot([ds.loc[(wl,p),"mean"] for p in POLICY_ORDER],
            [es.loc[(wl,p),"mean"]/e0 for p in POLICY_ORDER],color="0.7",lw=0.8,zorder=1)
    for p in POLICY_ORDER:
        ax.scatter(ds.loc[(wl,p),"mean"],es.loc[(wl,p),"mean"]/e0,marker=mk[p],zorder=2,
                   label=p if wl==WORKLOADS[0] else None)
ax.set_xlabel("Mean event delay (ms)"); ax.set_ylabel("Energy / baseline"); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig_energy_delay.png",dpi=150); plt.show()

fig,ax=plt.subplots(figsize=(6.5,4)); wls=[w for w in WORKLOADS if (w,"Baseline") in es.index]
x=np.arange(len(wls)); w=0.2
for i,p in enumerate(POLICY_ORDER):
    ax.bar(x+(i-1.5)*w,[es.loc[(wl,p),"mean"] for wl in wls],w,
           yerr=[es.loc[(wl,p),"ci95"] for wl in wls],capsize=3,label=p)
ax.set_xticks(x); ax.set_xticklabels([w.split()[0] for w in wls]); ax.set_ylabel("Energy (mJ)")
ax.legend(); ax.grid(axis="y",alpha=0.3); fig.tight_layout()
fig.savefig(f"{OUT_DIR}/fig_energy_bars.png",dpi=150); plt.show()
print("saved figures to",OUT_DIR)

## 9. Export LaTeX tables (to the output dir)

In [ ]:
def latex_results():
    L=["% auto-generated by uatimer_kaggle.ipynb",r"\begin{tabular}{llrr}",r"\toprule",
       r"Workload & Policy & Energy (mJ) & Delay (ms) \\",r"\midrule"]
    for wl in [w for w in WORKLOADS if (w,"Baseline") in es.index]:
        for j,p in enumerate(POLICY_ORDER):
            e=es.loc[(wl,p)]; d=ds.loc[(wl,p)]
            L.append(f"{wl if j==0 else ''} & {p} & {e['mean']:.0f}$\\pm${e['ci95']:.0f} "
                     f"& {d['mean']:.0f}$\\pm${d['ci95']:.0f} \\\\")
        L.append(r"\midrule")
    L[-1]=r"\bottomrule"; L.append(r"\end{tabular}"); return "\n".join(L)
open(f"{OUT_DIR}/table_results.tex","w").write(latex_results())
replay_df.to_csv(f"{OUT_DIR}/replay_behaviour.csv",index=False)
print(latex_results()[:500]); print("\nwrote",OUT_DIR+"/table_results.tex, replay_behaviour.csv")

## Summary

- Attach a Kaggle Dataset with `raw_runs.csv` (≥10 runs/cell) → the
  notebook switches to real data automatically and every table, statistic,
  figure, and LaTeX export reflects only your measurements.
- Without it, the pipeline runs on clearly-labelled synthetic data so you
  can verify it works.
- Outputs land in `/kaggle/working/` (Output tab): figures, LaTeX tables,
  and the control-behaviour replay CSV.

Nothing here is fabricated when real data is supplied. The energy replay is
a control-logic model and never claims device-energy numbers.